# Free Manga Translator — Kaggle GPU backend

Runs the 8-step manga translation pipeline on Kaggle's free NVIDIA T4 GPU and exposes it over an HTTPS tunnel so your Chrome extension (running locally in your browser) can reach it.

**Full documentation:** `docs/KAGGLE_DEPLOYMENT.md` in this repository — read it before your first run. This notebook mirrors that document's cell-by-cell walkthrough exactly.

**Before running anything:**
1. Side panel → Accelerator → **GPU T4 x2**.
2. Side panel → Internet → **On**.
3. Add-ons → Data → attach your private `fmt-core-pipeline` dataset.
4. Add-ons → Secrets → attach the secrets listed in `docs/KAGGLE_DEPLOYMENT.md` §4: `NGROK_AUTHTOKEN` and `FMT_AUTH_TOKEN` always, plus one per provider you actually use, plus `NGROK_STATIC_DOMAIN` (your claimed domain) — creating this one is what lets **Run All** work with zero manual cell edits.
5. All cells below are idempotent — safe to re-run individually if something fails partway. If the whole session dies, just re-run Cell 1 through Cell 6 top to bottom.

**Using "Run All":** with the secrets above (including `NGROK_STATIC_DOMAIN`) attached, **Run → Run All** will carry Cells 1-6 through to a working public tunnel with no manual editing required. It will then appear to hang on **Cell 7** — this is correct and intentional, not a bug: Cell 7 is a deliberate infinite keep-alive loop with no natural end. Let it run for as long as you're using the extension; when you're done, click the ■ Stop button on Cell 7, then run Cell 8 by hand to shut down cleanly.

## Cell 1 — environment sanity check
Confirms the GPU is actually visible, the dataset is attached, and prints every preinstalled library version Cell 2 is about to touch (torch/numpy/cv2/onnxruntime) via a fresh subprocess — never a kernel import, since Cell 2 uninstalls/reinstalls several of these and an already-imported module would linger stale in this kernel's memory. Kaggle mounts an attached dataset at `/kaggle/input/<slug>` in most sessions, but interactive/draft sessions have been observed mounting it one level deeper at `/kaggle/input/datasets/<your-username>/<slug>` instead — this cell checks both so it works either way.

In [ ]:
# fmt-cell1-v3
import subprocess, os, sys, time

_FMT_T0 = time.time()  # shared timeline start -- later cells print elapsed-since-here

DATASET_SLUG = "fmt-core-pipeline"

def find_dataset_dir(slug):
    candidates = [f"/kaggle/input/{slug}"]
    datasets_root = "/kaggle/input/datasets"
    if os.path.isdir(datasets_root):
        candidates += [
            f"{datasets_root}/{owner}/{slug}" for owner in os.listdir(datasets_root)
        ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    raise AssertionError(
        f"Dataset not attached -- Add-ons -> Data -> attach your {slug} dataset "
        f"(checked: {candidates})"
    )

DATASET_DIR = find_dataset_dir(DATASET_SLUG)
print("Dataset found at:", DATASET_DIR)
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
    capture_output=True, text=True,
).stdout)

_preflight = subprocess.run(
    [sys.executable, "-c", (
        "import importlib.util as u\n"
        "for name in ('torch', 'numpy', 'cv2', 'onnxruntime', 'onnxruntime_gpu'):\n"
        "    spec = u.find_spec(name)\n"
        "    if spec is None:\n"
        "        print(f'{name}: not installed'); continue\n"
        "    try:\n"
        "        mod = __import__(name)\n"
        "        print(f'{name}: {getattr(mod, \"__version__\", \"?\")}')\n"
        "    except Exception as e:\n"
        "        print(f'{name}: import failed -- {e}')\n"
        "import os\n"
        "print('DejaVuSans-Bold present:', os.path.exists('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'))\n"
    )],
    capture_output=True, text=True,
)
print(_preflight.stdout)
if _preflight.returncode != 0:
    print(_preflight.stderr)

## Cell 2 — copy code, fonts, CORS + tokenizer-routing hot-patches, smart dependency install
`/kaggle/input` is read-only, so the code is copied to `/kaggle/working` first, from `DATASET_DIR` (resolved by Cell 1). This cell also: (1) hot-patches the *copied* `main.py`'s CORS config to allow ngrok's bypass header, and the *copied* `ml_region_lib.py` with a tokenizer-routing shim fixing magi's TrOCR tokenizer on transformers ≥5.13 (§4/§5 explain both — temporary until the next dataset version bakes them in); (2) downloads and validates the two vendored ComicNeue fonts, since the dataset zip doesn't include them and step 8's Windows-first font paths need a Linux fallback in place *before* the backend process starts; (3) installs dependencies in an order that avoids the classic Kaggle dual-`cv2` and CPU-`onnxruntime`-shadowing traps, pins `onnxruntime-gpu`/`torchaudio` dynamically to whatever CUDA/torch build is actually active, and pins `transformers==5.12.0` — the exact version the full pipeline is proven on locally, daily, and the one where magi's TrOCR tokenizer load has been verified (both directions, on this machine, before this notebook shipped) to actually work. The smoke test at the end runs the literal `TrOCRProcessor.from_pretrained(...)` call magi makes, not just an import check, so a real failure surfaces here in seconds instead of during warmup minutes later. This cell is safe to re-run.

**Timing:** every step now prints a `[TIMING]` line with its own duration and a running total (Cells 4/5/5b/6 print elapsed-since-Cell-1 too), so a cold `Run All` tells us exactly where the time goes instead of guessing — see `docs/KAGGLE_DEPLOYMENT.md` §11 for what to do with these numbers.

In [ ]:
# fmt-cell2-v8
import os, re, shutil, subprocess, sys, time, urllib.request

# --- Timing instrumentation. Purely additive -- no behavior change. Prints how
# long each named step took plus a running total, so the FIRST genuinely cold
# Run All tells us where Cell 2's time actually goes (pip resolve/download vs.
# CPU-bound wheel unpacking vs. something else) instead of guessing.
_cell2_t0 = time.time()
_lap_prev = [_cell2_t0]
def _lap(label):
    now = time.time()
    since_t0 = now - globals().get("_FMT_T0", _cell2_t0)
    print(f"[TIMING] {label}: {now - _lap_prev[0]:.1f}s (cell2 running total: {now - _cell2_t0:.1f}s, since Cell 1: {since_t0:.1f}s)")
    _lap_prev[0] = now

WORK = "/kaggle/working/core_pipeline"
shutil.copytree(os.path.join(DATASET_DIR, "core_pipeline"), WORK, dirs_exist_ok=True)
_lap("copy repo from dataset")

# --- 1. Hot-patch the COPY's CORS config so ngrok's free-tier browser-warning
# bypass header is allowed (see docs/KAGGLE_DEPLOYMENT.md §4). Temporary: bake
# this into the next dataset version and remove the patch.
_main_py = os.path.join(WORK, "backend_api", "app", "main.py")
with open(_main_py, "r", encoding="utf-8") as f:
    _main_src = f.read()
_old_headers = 'allow_headers=["Content-Type", "Authorization", "X-Fmt-Client", "X-Fmt-Auth"]'
_new_headers = 'allow_headers=["Content-Type", "Authorization", "X-Fmt-Client", "X-Fmt-Auth", "ngrok-skip-browser-warning"]'
if _old_headers in _main_src:
    _main_src = _main_src.replace(_old_headers, _new_headers)
    with open(_main_py, "w", encoding="utf-8") as f:
        f.write(_main_src)
    print("CORS hot-patch applied.")
elif "ngrok-skip-browser-warning" in _main_src:
    print("CORS hot-patch already present (repo copy is up to date).")
else:
    raise AssertionError("Expected CORS allow_headers line not found -- main.py has changed shape, patch manually")

# --- 1b. Hot-patch the COPY's ml_region_lib.py with the tokenizer-routing shim.
# transformers >=5.13.0 registers TrOCR's model_type ("vision-encoder-decoder",
# loaded internally by magi via TrOCRProcessor) in TOKENIZER_MAPPING_NAMES pointing
# at the generic TokenizersBackend class, which can only build from a tokenizer.json
# -- and microsoft/trocr-base-printed has never shipped one (only vocab.json+
# merges.txt). AutoTokenizer.from_pretrained then raises a misleading "need
# sentencepiece or tiktoken" ValueError that has nothing to do with either package.
# Verified empirically (both directions) before this notebook was written: this
# exact patch fixes transformers 5.14.1 and is a harmless no-op on 5.12.0. The
# dataset is stale until re-uploaded, so this hot-patches the COPY the same way the
# CORS patch above does; becomes a no-op once a new dataset version bakes the repo
# fix (ml_region_lib.py's load_semantic_model) in directly.
_ml_region_lib_py = os.path.join(WORK, "python", "common", "ml_region_lib.py")
with open(_ml_region_lib_py, "r", encoding="utf-8") as f:
    _mrl_src = f.read()
_tok_shim_marker = "Tokenizer-routing shim"
_tied_weights_anchor = '        transformers.PreTrainedModel.all_tied_weights_keys = property(get_tied, set_tied)\n\n    repo_id = model_path or "ragavsachdeva/magi"'
_tok_shim_code = (
    '        transformers.PreTrainedModel.all_tied_weights_keys = property(get_tied, set_tied)\n\n'
    '    # Tokenizer-routing shim (see docs/KAGGLE_DEPLOYMENT.md §5/§10 for the full story).\n'
    '    from transformers.models.auto import tokenization_auto as _tok_auto\n'
    '    _tok_auto.TOKENIZER_MAPPING_NAMES["vision-encoder-decoder"] = "RobertaTokenizer"\n\n'
    '    repo_id = model_path or "ragavsachdeva/magi"'
)
if _tok_shim_marker in _mrl_src:
    print("Tokenizer-routing shim already present (repo copy is up to date).")
elif _tied_weights_anchor in _mrl_src:
    _mrl_src = _mrl_src.replace(_tied_weights_anchor, _tok_shim_code, 1)
    with open(_ml_region_lib_py, "w", encoding="utf-8") as f:
        f.write(_mrl_src)
    print("Tokenizer-routing shim applied.")
else:
    raise AssertionError("Expected tied-weights anchor not found in ml_region_lib.py -- file has changed shape, patch manually")
_lap("CORS + tokenizer-routing hot-patches")

# --- 2. Vendored fonts (dataset ships none; step 8 needs a Linux-reachable font
# before backend import). Download + validate; fall back to DejaVu on failure.
from PIL import ImageFont

FONTS_DIR = "/kaggle/working/fonts"
os.makedirs(FONTS_DIR, exist_ok=True)
_font_urls = {
    "ComicNeue-Bold.ttf": "https://github.com/google/fonts/raw/main/ofl/comicneue/ComicNeue-Bold.ttf",
    "ComicNeue-Regular.ttf": "https://github.com/google/fonts/raw/main/ofl/comicneue/ComicNeue-Regular.ttf",
}
_fonts_ok = True
for _name, _url in _font_urls.items():
    _dest = os.path.join(FONTS_DIR, _name)
    try:
        urllib.request.urlretrieve(_url, _dest)
        if os.path.getsize(_dest) < 50_000:
            raise ValueError(f"downloaded file too small ({os.path.getsize(_dest)} bytes)")
        ImageFont.truetype(_dest, 24)  # raises on a corrupt/truncated font file
        print(f"{_name}: downloaded and validated.")
    except Exception as e:
        if os.path.exists(_dest):
            os.remove(_dest)
        print(f"{_name}: download/validation failed ({e}) -- will fall back to DejaVu if present.")
        _fonts_ok = False

_dejavu = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
if not _fonts_ok and not os.path.exists(_dejavu):
    raise AssertionError(
        "ComicNeue fonts failed to download AND DejaVu fallback is absent -- "
        "step 8 typesetting would render unreadable bitmap text. Check internet access."
    )
_lap("fonts download+validate")

# --- 3. Torch: keep the preinstalled build if it's already CUDA-capable >= 2.6.
# Kaggle's image version drifts over time (observed torch 2.6.0+cu124 one day,
# 2.10.0+cu128 the next) -- a naive string .startswith(("2.6",...,"2.9")) check
# silently breaks on any double-digit minor version (e.g. "2.10"), so compare
# the (major, minor) tuple numerically instead.
import torch
_torch_ver_match = re.match(r"(\d+)\.(\d+)", torch.__version__)
_torch_major_minor = tuple(int(x) for x in _torch_ver_match.groups()) if _torch_ver_match else (0, 0)
_keep_preinstalled_torch = _torch_major_minor >= (2, 6) and torch.cuda.is_available()
if _keep_preinstalled_torch:
    print(f"Reusing preinstalled torch {torch.__version__} (CUDA available: True)")
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "torch==2.6.0", "torchvision==0.21.0",
         "--index-url", "https://download.pytorch.org/whl/cu124"],
        check=True,
    )
_lap("torch check/install")

# --- 4. Uninstall anything that could shadow/conflict before installing fresh.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y",
     "onnxruntime", "opencv-python", "opencv-python-headless", "opencv-contrib-python"],
    check=False,
)
_lap("uninstall shadow packages")

# --- 5. Filtered requirements install: drop torch/torchvision pins when keeping
# the preinstalled build, swap opencv-python for the headless build up front.
def _filter_requirements(src_path, dest_path, drop_torch):
    with open(src_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    out = []
    for line in lines:
        stripped = line.strip()
        low = stripped.lower()
        if drop_torch and (low.startswith("torch==") or low.startswith("torchvision==")):
            continue
        if low.startswith("opencv-python==") or low.startswith("opencv-python>="):
            out.append(line.replace("opencv-python", "opencv-python-headless", 1))
            continue
        out.append(line)
    with open(dest_path, "w", encoding="utf-8") as f:
        f.writelines(out)

_filtered_reqs = "/kaggle/working/requirements.kaggle.txt"
_filter_requirements(os.path.join(WORK, "python", "requirements.txt"), _filtered_reqs, _keep_preinstalled_torch)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", _filtered_reqs], check=True)
_lap("filtered requirements.txt install (paddle/paddleocr/easyocr/ultralytics/etc)")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", os.path.join(WORK, "backend_api", "requirements.txt")],
    check=True,
)
_lap("backend_api requirements.txt install")
subprocess.run([sys.executable, "-m", "pip", "install", "pyngrok"], check=True)
_lap("pyngrok install")

# --- 5b. sentencepiece + protobuf: needed by the NLLB local-translator tokenizer
# (facebook/nllb-200-distilled-600M) -- unrelated to the TrOCR/magi tokenizer issue,
# which is fixed by the routing shim above (step 1b), not by these packages.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall",
     "sentencepiece>=0.2.0", "protobuf>=3.20.0,<6"],
    check=True,
)
_lap("sentencepiece+protobuf reinstall")

# --- 6. Post-install cleanup: some deps (ultralytics, paddleocr/paddlex) pull
# non-headless opencv back in transitively. Force headless-only, last, no-deps.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "opencv-python", "opencv-contrib-python"],
    check=False,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", "opencv-python-headless>=4.10"],
    check=True,
)
_lap("opencv-headless cleanup+reinstall")

# --- 6b. onnxruntime-gpu>=1.26.0 resolves to 1.27.0+, which dropped CUDA 12 as
# the default and now needs libcudart.so.13 -- absent on any Kaggle CUDA-12.x box.
# Pin the last version that still defaults to CUDA 12 (compatible across all
# CUDA 12.x minor versions per NVIDIA's own minor-version-compat guidelines, so
# this is safe whether Kaggle's image is on cu124, cu128, or another 12.x build).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", "onnxruntime-gpu==1.26.0"],
    check=True,
)
_lap("onnxruntime-gpu pin")

# --- 6c. torchaudio: some transitive dep (transformers/easyocr audio extras)
# pulls in a torchaudio build that doesn't match the ACTIVE torch, breaking with
# "undefined symbol" at import time during warmup. Never trust this kernel's
# `torch` import for the version here -- it can be stale if step 3 reinstalled
# torch in a separate process. Query the real active version via a fresh
# subprocess, then force a torchaudio build matching that exact torch+CUDA
# combo, no-deps so it can't drag a different torch back in.
_torch_active = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, check=True,
).stdout.strip()
_torch_base, _, _torch_local = _torch_active.partition("+")
_cuda_tag = _torch_local if _torch_local.startswith("cu") else "cu124"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps",
     f"torchaudio=={_torch_base}", "--index-url", f"https://download.pytorch.org/whl/{_cuda_tag}"],
    check=True,
)
_lap("torchaudio pin")

# --- 6d. transformers pin. >=5.13.0 has the tokenizer-routing regression fixed by
# the hot-patched shim above (step 1b) -- this pin is belt-and-braces, matching the
# exact version the full pipeline (magi + TrOCR + manga-ocr + NLLB) is proven on
# daily on the maintainer's own machine. NOT --no-deps: pip must co-resolve
# tokenizers/huggingface-hub/safetensors consistently with this exact version.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "transformers==5.12.0"],
    check=True,
)
_lap("transformers pin")

# --- 7. Fresh-subprocess dist scan + real CUDA-provider smoke test, PLUS an
# actual load of the exact tokenizer that has been failing during warmup -- the
# full TrOCRProcessor call magi itself makes (image processor + tokenizer), not
# just a bare AutoTokenizer check. A fresh subprocess is required because this
# kernel may still hold stale imports of packages we just uninstalled/reinstalled.
_model_a_path = os.path.join(WORK, "models", "comictextdetector.pt.onnx")
_smoke_script = f"""
import importlib.metadata as md
dists = {{d.metadata['Name'].lower() for d in md.distributions()}}
assert 'opencv-python' not in dists and 'opencv-contrib-python' not in dists, f"stray opencv dist present: {{dists & {{'opencv-python','opencv-contrib-python'}}}}"
assert 'opencv-python-headless' in dists, "opencv-python-headless missing"
assert 'onnxruntime' not in dists, "CPU onnxruntime dist present -- will shadow onnxruntime-gpu"
assert 'onnxruntime-gpu' in dists, "onnxruntime-gpu missing"

import numpy, cv2, PIL, torch, torchaudio, paddle, paddleocr, easyocr, ultralytics, sentencepiece, transformers, onnxruntime as ort
assert torch.cuda.is_available(), "torch.cuda.is_available() is False"

session = ort.InferenceSession({_model_a_path!r}, providers=['CUDAExecutionProvider'])
assert session.get_providers()[0] == 'CUDAExecutionProvider', session.get_providers()

from transformers import TrOCRProcessor
_trocr_proc = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
print('TrOCR processor (magi dependency) loaded OK:', type(_trocr_proc.tokenizer).__name__, type(_trocr_proc.image_processor).__name__)

print('numpy', numpy.__version__)
print('cv2', cv2.__version__)
print('torch', torch.__version__, 'cuda:', torch.version.cuda)
print('torchaudio', torchaudio.__version__)
print('paddle', paddle.__version__)
print('sentencepiece', sentencepiece.__version__)
print('transformers', transformers.__version__)
print('onnxruntime', ort.__version__, 'active provider:', session.get_providers()[0])
print('SMOKE TEST PASSED')
"""
_smoke = subprocess.run([sys.executable, "-c", _smoke_script], capture_output=True, text=True)
print(_smoke.stdout)
if _smoke.returncode != 0:
    print(_smoke.stderr)
    raise RuntimeError("Dependency smoke test failed -- see stderr above before continuing.")
_lap("smoke test (incl. TrOCRProcessor load -- watch this one for HF download time)")
print(f"[TIMING] Cell 2 TOTAL: {time.time() - _cell2_t0:.1f}s")

## Cell 3 — load secrets into environment
Reads each provider's key straight from its own small Kaggle Secret into `os.environ` — no `.env` file is ever written to disk. A provider whose secret was never created is simply left unconfigured (matches how an unfilled line in a local `.env` behaves — zero-quota, never reserved). `FMT_AUTH_TOKEN` is required and validated non-empty after stripping whitespace — a token that silently strips to `""` would leave every gated route armed only by the (public, in-repo) `X-Fmt-Client` header, so this cell refuses to continue rather than fail open. Non-secret pipeline tuning is hardcoded below since it isn't sensitive. **Nothing here is ever printed.**

In [ ]:
# fmt-cell3-v2
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def optional_secret(label):
    # Tolerates either failure mode for a secret that was never created (raises,
    # or returns None/empty) -- a provider you don't use just stays unconfigured,
    # exactly like an unfilled line in .env (already zero-quota, never reserved).
    try:
        return secrets.get_secret(label) or ""
    except Exception:
        return ""

PROVIDER_SECRET_LABELS = [
    "GEMINI_API_KEYS", "GITHUB_API_KEYS", "GROQ_API_KEYS", "MISTRAL_API_KEYS",
    "OPENROUTER_API_KEYS", "CEREBRAS_API_KEYS", "FIREWORKS_API_KEYS",
    "CLOUDFLARE_WORKERS_API_KEYS", "CLOUDFLARE_ACCOUNT_IDS", "NVIDIA_NIM_API_KEYS",
]
configured = []
for label in PROVIDER_SECRET_LABELS:
    value = optional_secret(label)
    if value:
        os.environ[label] = value
        configured.append(label.replace("_API_KEYS", "").replace("_ACCOUNT_IDS", ""))

_auth_token = optional_secret("FMT_AUTH_TOKEN").strip()
if not _auth_token:
    raise AssertionError(
        "FMT_AUTH_TOKEN secret is missing or empty -- Add-ons -> Secrets -> create it "
        "(see docs/KAGGLE_DEPLOYMENT.md §4 Step 3). Refusing to start with auth "
        "silently disabled on a public tunnel."
    )
os.environ["FMT_AUTH_TOKEN"] = _auth_token

# Non-secret pipeline tuning -- safe to hardcode, mirrors this repo's .env.example.
# Edit directly if you want different behavior on Kaggle than locally.
os.environ.update({
    "TRANSLATION_PROVIDER_ORDER": "mistral,cerebras,fireworks,groq,nvidia,github,openrouter,gemini,cloudflare",
    "USE_API_TRANSLATION": "auto",
    "LOCAL_TRANSLATOR_MODEL": "facebook/nllb-200-distilled-600M",
    "NVIDIA_KEY_ROLES": "translation,vision_ocr,backup",
    "OPENROUTER_KEY_ROLES": "translation,translation,vision_ocr,backup",
    "FMT_STARTUP_WARMUP": "0",
    "FMT_GPU_IDLE_UNLOAD_SECONDS": "600",
})
print(f"Configured providers ({len(configured)}/10): {', '.join(configured) or 'none'}")
print("Secrets loaded (values not shown).")

## Cell 4 — launch the backend
Runs uvicorn the same way `start_backend.ps1` does locally (same working directory, same module path), as a background subprocess. Bind stays on `127.0.0.1` -- the tunnel client in cell 6 runs on this same VM and reaches it over loopback, so there's never a reason to bind wider here. Re-run safe: terminates any backend this notebook already started before launching a new one, so a re-run doesn't leak a process holding port 8766.

In [ ]:
# fmt-cell4-v3
import subprocess, sys, time

if "backend_proc" in globals() and backend_proc.poll() is None:
    backend_proc.terminate()
    try:
        backend_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        backend_proc.kill()
subprocess.run(["pkill", "-f", "uvicorn backend_api.app.main:app"], check=False)
time.sleep(1)

backend_log = open("/kaggle/working/backend.log", "w")
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend_api.app.main:app",
     "--host", "127.0.0.1", "--port", "8766"],
    cwd="/kaggle/working/core_pipeline",
    stdout=backend_log, stderr=subprocess.STDOUT,
)
print(f"Backend starting, pid={backend_proc.pid}. Logs: /kaggle/working/backend.log")
print(f"[TIMING] since Cell 1 start: {time.time() - globals().get('_FMT_T0', time.time()):.1f}s")

## Cell 5 — wait for health + force warmup
Polls `/v1/health` until the process accepts connections, checking after every poll whether the backend process has actually died (rather than waiting out the full timeout for nothing) and printing the log tail immediately if it has. Then forces `/v1/warmup` and polls for up to 20 minutes, since a first-ever run downloads several GB (magi, manga-ocr, easyocr, the ~2.4GB NLLB translator) — see `docs/KAGGLE_DEPLOYMENT.md` §7. Requests now send `X-Fmt-Auth` alongside `X-Fmt-Client`, since `/v1/warmup` is one of the auth-gated routes once `FMT_AUTH_TOKEN` is set (Cell 3 always sets it).

In [ ]:
# fmt-cell5-v3
import time, urllib.request, urllib.error, json

def _print_log_tail(n=60):
    try:
        with open("/kaggle/working/backend.log", "r", errors="replace") as f:
            lines = f.readlines()
        print(f"--- last {min(n, len(lines))} lines of backend.log ---")
        print("".join(lines[-n:]))
    except FileNotFoundError:
        print("(backend.log not found)")

def get_json(url, method="GET"):
    req = urllib.request.Request(
        url, method=method,
        headers={
            "X-Fmt-Client": "free-manga-translator-extension",
            "X-Fmt-Auth": os.environ.get("FMT_AUTH_TOKEN", ""),
        },
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.load(resp)

_warmup_t0 = time.time()
_health_deadline = time.time() + 300  # 5 minutes
while time.time() < _health_deadline:
    if backend_proc.poll() is not None:
        print(f"Backend process exited early (code {backend_proc.returncode}).")
        _print_log_tail()
        raise RuntimeError("Backend process died before health check passed -- see log tail above.")
    try:
        get_json("http://127.0.0.1:8766/v1/health")
        break
    except Exception:
        time.sleep(2)
else:
    _print_log_tail()
    raise RuntimeError("Backend did not come up within 5 minutes -- see log tail above.")

print(f"Health check passed. [TIMING] {time.time() - _warmup_t0:.1f}s since backend launch, "
      f"{time.time() - globals().get('_FMT_T0', time.time()):.1f}s since Cell 1 start")

get_json("http://127.0.0.1:8766/v1/warmup", method="POST")
_warmup_deadline = time.time() + 1200  # 20 minutes -- first run downloads several GB
_last_status = None
while time.time() < _warmup_deadline:
    if backend_proc.poll() is not None:
        print(f"Backend process exited during warmup (code {backend_proc.returncode}).")
        _print_log_tail()
        raise RuntimeError("Backend process died during warmup -- see log tail above.")
    status = get_json("http://127.0.0.1:8766/v1/health")["warmup"]["status"]
    if status != _last_status:
        print(time.strftime("%H:%M:%S"), "warmup:", status,
              f"[TIMING] {time.time() - _warmup_t0:.1f}s since backend launch")
        _last_status = status
    if status == "pass":
        break
    if status == "fail":
        _print_log_tail()
        raise RuntimeError("Warmup reported 'fail' -- see log tail above.")
    time.sleep(5)
else:
    _print_log_tail()
    raise RuntimeError("Warmup did not complete within 20 minutes -- see log tail above.")

print(f"[TIMING] Total warmup time: {time.time() - _warmup_t0:.1f}s. "
      f"Since Cell 1 start: {time.time() - globals().get('_FMT_T0', time.time()):.1f}s")

## Cell 5b — loopback end-to-end translate (new)
Runs one real translation entirely over `127.0.0.1`, before any tunnel exists. This is the single highest-value check in the whole notebook: it exercises the CUDA ONNX text detector, OpenCV, PaddleOCR, EasyOCR, the LaMa inpainter, the fonts installed in Cell 2, and whichever translation provider keys you configured — all in one request. If this cell passes, the pipeline itself is proven working and any remaining failure is tunnel/extension-side, not pipeline-side.

In [ ]:
# fmt-cell5b-v3
import base64, json, time, urllib.request
from IPython.display import Image, display

_sample_path = "/kaggle/working/core_pipeline/samples/sample1/sample.jpg"
with open(_sample_path, "rb") as f:
    _sample_b64 = base64.b64encode(f.read()).decode("ascii")

_payload = json.dumps({
    "imageData": _sample_b64,
    "sourceLanguage": "ja",
    "targetLanguage": "en",
    "qualityProfile": "strict",
}).encode("utf-8")

_req = urllib.request.Request(
    "http://127.0.0.1:8766/v1/translate-image",
    data=_payload, method="POST",
    headers={
        "Content-Type": "application/json",
        "X-Fmt-Client": "free-manga-translator-extension",
        "X-Fmt-Auth": os.environ.get("FMT_AUTH_TOKEN", ""),
    },
)
_translate_t0 = time.time()
# 600s, not 300s: Kaggle's CPU is the long pole for the CPU-bound paddlepaddle OCR
# pass on a first (cold-cache) translate -- a tighter timeout here would falsely
# fail an otherwise-working pipeline before it even finishes one real page.
with urllib.request.urlopen(_req, timeout=600) as resp:
    _result = json.load(resp)
_translate_elapsed = time.time() - _translate_t0

print("status:", _result.get("status"))
_report = _result.get("report", {})
print("report:", json.dumps(_report, indent=2)[:1000])
if _result.get("status") != "pass":
    raise RuntimeError(f"Loopback translate did not pass: {_result.get('error')}")

# Which translation path was actually used matters for timing: local NLLB fallback
# downloads ~2.4GB on a cold cache (not part of warmup), while an API provider
# doesn't -- report.json's provider-ish fields tell us which happened, so we know
# on THIS run whether that download was in the timing or not. Field name isn't
# fully pinned down, so try a few likely spots and just show "unknown" (with the
# report already printed above) rather than guessing wrong.
_translation_block = _report.get("translation")
_provider_used = (
    _report.get("translationProvider")
    or _report.get("provider")
    or (_translation_block.get("provider") if isinstance(_translation_block, dict) else None)
)
print(f"[TIMING] Loopback translate: {_translate_elapsed:.1f}s "
      f"(provider used: {_provider_used or 'unknown -- check report above for the field name'}) "
      f"[TIMING] since Cell 1 start: {time.time() - globals().get('_FMT_T0', time.time()):.1f}s")

_out_data_url = _result.get("translatedImageDataUrl") or _result.get("imageDataUrl")
if _out_data_url and "," in _out_data_url:
    _out_bytes = base64.b64decode(_out_data_url.split(",", 1)[1])
    display(Image(data=_out_bytes))
print("Loopback translate PASSED -- pipeline is proven working end to end.")

## Cell 6 — open the tunnel
Uses `pyngrok` with your claimed static domain, so the public URL never changes between sessions. Re-run safe: kills any tunnel this notebook already opened first (the free tier only allows one agent session, so a bare re-run of `ngrok.connect` on the same domain fails). **For a true "Run All" with no manual editing**, create the optional `NGROK_STATIC_DOMAIN` secret (§4) with your claimed domain as the value — this cell reads it automatically. If that secret is missing, it falls back to the `STATIC_DOMAIN` constant below, which needs manual editing. If ngrok rejects the explicit domain as a "custom subdomain" (a known quirk with newer `.ngrok-free.dev` dev-domains, even when the domain is genuinely yours — see §10), this cell automatically retries with no domain argument at all, letting ngrok auto-assign your account's dev domain instead of hard-failing.

In [ ]:
# fmt-cell6-v4
import time
from pyngrok import ngrok, conf
from pyngrok.exception import PyngrokNgrokHTTPError

secrets = UserSecretsClient()
conf.get_default().auth_token = secrets.get_secret("NGROK_AUTHTOKEN")

try:
    ngrok.kill()
except Exception:
    pass

# EDIT ME: the exact static domain you claimed in the ngrok dashboard, used only
# if you didn't create an optional NGROK_STATIC_DOMAIN secret.
STATIC_DOMAIN = "yourname-something.ngrok-free.app"

_domain = optional_secret("NGROK_STATIC_DOMAIN") or STATIC_DOMAIN
if _domain == "yourname-something.ngrok-free.app":
    raise AssertionError(
        "STATIC_DOMAIN is still the placeholder -- edit it to your claimed ngrok "
        "domain, or create an NGROK_STATIC_DOMAIN secret, before running this cell."
    )

try:
    tunnel = ngrok.connect(8766, domain=_domain)
except PyngrokNgrokHTTPError as e:
    # ngrok's newer .ngrok-free.dev "dev domains" have been observed rejecting an
    # explicit domain= as a paid-only "custom subdomain" even when it's genuinely
    # reserved to this account. Fall back to auto-assignment rather than hard-fail
    # -- the account's own dev domain is picked automatically with no argument.
    if "custom subdomain" in str(e).lower():
        print(f"ngrok rejected the explicit domain ({_domain}) as a 'custom subdomain' "
              "-- falling back to auto-assigned dev domain.")
        tunnel = ngrok.connect(8766)
    else:
        raise

print("Public URL:", tunnel.public_url)
print("Paste this into the extension popup's Local Pipeline URL field:")
print(f"  {tunnel.public_url}/v1/translate-image")
print(f"[TIMING] Total time to working public URL, since Cell 1 start: "
      f"{time.time() - globals().get('_FMT_T0', time.time()):.1f}s")

## Cell 7 — keep the session alive
Kaggle interactive sessions can idle-disconnect. This cell keeps the notebook actively running and prints a periodic health check. **This is an intentional infinite loop** -- interrupt it (■ Stop) when you're done reading and want to move to the shutdown cell. **Important:** this loop does NOT defeat Kaggle's ~60-minute idle watchdog — a running cell is not counted as user interaction, only clicks/keystrokes in the tab are. Keep the tab open and interact with it roughly hourly, or the session will still be killed even while this cell is running. Don't run this cell during initial setup/verification — only start it at handoff, once cells 1-6 have all passed.

In [ ]:
import time

while True:
    try:
        status = get_json("http://127.0.0.1:8766/v1/health")
        print(
            time.strftime("%H:%M:%S"), "ok, warmup:", status["warmup"]["status"],
            "active jobs:", status["scheduler"]["active"],
        )
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "backend check failed:", e)
    time.sleep(60)

## Cell 8 — clean shutdown
Closes the tunnel and stops the backend. Uses `ngrok.kill()` rather than `ngrok.disconnect(tunnel.public_url)` since the latter fails with a `NameError` if the kernel was ever restarted and `tunnel` no longer exists — `ngrok.kill()` works regardless. Nothing is written to disk by Cell 3 in this design, so there's no leftover secrets file to clean up.

In [ ]:
# fmt-cell8-v2
try:
    ngrok.kill()
except Exception:
    pass
if "backend_proc" in globals() and backend_proc.poll() is None:
    backend_proc.terminate()
print("Shut down cleanly.")